# 📚 Learning Companion — Whole Workflow Walkthrough

**Track D: Personalized Tutoring & Adaptive Learning (LangChain + LangGraph)**

End-to-end pipeline:

```
G0 INGEST (pdf/slides/video_txt) → Chroma vector store + NetworkX concept graph
G1 TUTOR (retrieve → grade → expand → generate → verify) with citations + refusal gate
G2 QUIZ (scope → generate → verify → feedback → dedup)
G3 LEARNER (BKT mastery per student x topic + forgetting + difficulty adaptation)
EVAL (simulator + RAGAS stub) → FastAPI + Streamlit
```

> Run with the **langchain_env** kernel (Python 3.11). No API keys needed — every LLM call has an offline fallback.

## 0. Setup — paths, imports, env
We add the project root to `sys.path` so `src.*` imports work from the notebook.

In [ ]:
import sys, os, json, pathlib
ROOT = "/home/preritubuntu/Multimodal_hack_proj"
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("cwd:", os.getcwd())
from src.config.settings import get_settings
s = get_settings()
print(f"EMBEDDING={s.EMBEDDING_MODEL} | VECTOR={s.VECTOR_BACKEND} | DIR={s.CHROMA_DIR}")
print(f"GENERATOR={s.GENERATOR_PROVIDER}/{s.GENERATOR_MODEL} | VERIFIER={s.VERIFIER_PROVIDER}/{s.VERIFIER_MODEL}")
print(f"TOP_K={s.TOP_K} CHUNK={s.CHUNK_SIZE}/{s.CHUNK_OVERLAP}")
print("has GOOGLE key:", bool(s.GOOGLE_API_KEY), "| has OPENAI key:", bool(s.OPENAI_API_KEY))

## 1. Project map — what each folder does

| Path | Role |
|---|---|
| `src/config/settings.py` | single `Settings`: models, embeddings, thresholds, DB paths |
| `src/schemas/models.py` | `CitationUnit` (every chunk), `QuizQuestion`, States |
| `src/chains/models.py` | **only** place creating LLMs/embeddings |
| `src/chains/prompts.py` | TAG / ANSWER / VERIFY / QUIZ `ChatPromptTemplate`s |
| `src/stores/vector.py` | Chroma via LangChain + `hybrid_search` + keyword re-rank |
| `src/stores/graph.py` | NetworkX concept/prereq graph |
| `src/stores/learner.py` | Custom BKT per student x topic |
| `src/graphs/ingest_graph.py` | **G0**: `load→split→tag→upsert→graph` |
| `src/graphs/tutor_graph.py` | **G1**: `retrieve→grade→expand→generate→verify`, checkpointed per student |
| `src/graphs/quiz_graph.py` | **G2**: `scope→generate→verify→feedback→BKT` |
| `src/eval/evaluate.py` | RAGAS wrapper + `simulate()` |
| `src/api/main.py` | FastAPI: `/ingest /chat /quiz /mastery/{id}` |
| `src/ui/app.py` | Streamlit: Upload | Tutor | Quiz | Dashboard |

In [ ]:
import pathlib
for p in sorted(pathlib.Path("src").rglob("*.py")):
    print(p)

## 2. Schemas — `CitationUnit` is the heart of grounding
Every chunk in Chroma carries this metadata so answers cite deep links: PDF `#page=n`, slides `#slide=n`, video `?t=Ns`.

In [ ]:
from src.schemas.models import CitationUnit, QuizQuestion
u = CitationUnit(topic="photosynthesis", concept="chlorophyll", content_type="textbook", source_file="bio.pdf", page=3, difficulty="easy")
print("deep_link:", u.deep_link())
print("cite:     ", u.cite())
v = CitationUnit(topic="photosynthesis", concept="light", content_type="video", source_file="lec1.mp4", t_start=95, t_end=150)
print("video link:", v.deep_link(), "|", v.cite())

## 3. Model layer (LangChain only) + prompts
Rule: **no raw SDK calls** — everything via `src/chains/models.py`. Without API keys, tutor/quiz catch the exception and use offline fallbacks.

In [ ]:
from src.chains import prompts as P
for name in ["TAG_PROMPT", "ANSWER_PROMPT", "VERIFY_PROMPT", "QUIZ_PROMPT"]:
    msgs = getattr(P, name).format_messages(**{k: f"<{k}>" for k in getattr(P, name).input_variables})
    print("="*15, name, "="*15)
    print(msgs[0].content[:300])
    print()

## 4. G0 Ingestion — `load → split → tag → upsert → graph`

- **load**: PPTX→1 doc/slide (keeps `slide`), TXT/SRT→video doc (`t_start`), PDF→Docling (`page`) else pypdf
- **split**: slides stay whole; else `RecursiveCharacterTextSplitter(900/135)`
- **tag**: LLM `CitationUnit` if keys exist, else heuristic defaults
- **upsert**: `Document(text+metadata)` → Chroma
- **graph**: `concept ← prereq(subtopic)` edges into NetworkX

In [ ]:
from pathlib import Path
from src.config.paths import upload_path
from src.graphs.ingest_graph import build_ingest_graph, ingest_file
g0 = build_ingest_graph()
print(g0.get_graph().draw_ascii())
sample = upload_path("sample.txt")
Path(sample).write_text(
    "Photosynthesis converts light energy into chemical energy in chloroplasts. "
    "Chlorophyll absorbs red and blue light. The Calvin cycle fixes carbon dioxide into glucose.")
out = ingest_file(sample)
print("kind:", out["kind"], "| units:", len(out["units"]), "| errors:", out["errors"])
print(json.dumps(out["units"][0], indent=2, default=str)[:800])

## 5. Knowledge base — check Chroma + concept graph
What did G0 actually store?

In [ ]:
from src.stores.vector import hybrid_search
from src.stores.graph import get_graph
docs = hybrid_search("photosynthesis chlorophyll", k=3)
print(f"hybrid_search returned {len(docs)} docs")
for d in docs:
    print("-", d.metadata, "|", d.page_content[:120])
G = get_graph()
print(f"\nconcept graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(list(G.edges(data=True))[:5])

## 6. G1 Tutor — `retrieve → grade → expand → generate → verify`

- **retrieve**: `hybrid_search(question, TOP_K)`
- **grade**: keyword-overlap gate; `best < 0.15 → refuse`
- **expand**: add prereqs from concept graph
- **generate**: LLM answer with citations, else extractive fallback personalized by BKT (`Simply put` / `In depth`)
- **verify**: second-vendor LLM SUPPORTED/NOT_SUPPORTED check
- Checkpointed in sqlite per `thread_id=student_id`

In [ ]:
from src.graphs.tutor_graph import build_tutor_graph, ask
g1 = build_tutor_graph()
print(g1.get_graph().draw_ascii())
r = ask("What is photosynthesis?", student_id="notebook-demo")
print("VERDICT:", r["verdict"])
print("CITATIONS:", r["citations"])
print("ANSWER:\n", r["answer"][:1000])

In [ ]:
# Refusal gate: off-material question must decline, not hallucinate
r2 = ask("Who won the 2026 football world cup?", student_id="notebook-demo")
print("VERDICT (expect refuse):", r2["verdict"])
print(r2["answer"])

## 7. G2 Quiz — `scope → generate → verify → feedback → BKT`

- **scope**: `difficulty=auto` → `difficulty_for(student, scope)` from BKT
- **generate**: retrieve scope context → LLM quiz chain, else offline fallback (one Q per sentence)
- **verify/dedup**: sha-hash dedup vs `questions.jsonl`; sympy check for numerical
- **feedback** (`n_feedback`): grading → `learner.update()` → weak-topic list + cited explanation

In [ ]:
from src.graphs.quiz_graph import build_quiz_graph, n_feedback
g2 = build_quiz_graph()
print(g2.get_graph().draw_ascii())
q = g2.invoke({"student_id": "notebook-demo", "scope": "photosynthesis",
               "difficulty_target": "easy", "n_questions": 2,
               "questions": [], "results": [], "weak_topics": []})
print(f"generated {len(q['questions'])} questions")
for item in q["questions"]:
    print(f"\n[{item['qhash']}] ({item['difficulty']}) {item['stem']}")
    print(f"   key={item['answer_key'][:100]} | src={item['source']}")

In [ ]:
# Grade: answer Q1 correctly, leave rest blank -> see BKT + weak topics
answers = {q["questions"][0]["qhash"]: q["questions"][0]["answer_key"]} if q["questions"] else {}
fb = n_feedback({**q, "student_id": "notebook-demo", "scope": "photosynthesis"}, answers=answers)
print(json.dumps(fb["results"], indent=2)[:1500])
print("weak_topics:", fb["weak_topics"])

## 8. G3 Learner — BKT per student x topic
Mastery update in `src/stores/learner.py`: posterior given observation, then `P_new = P_obs + (1-P_obs)*P_LEARN` with daily decay 0.995. Difficulty: `<0.4 easy`, `>0.75 hard`, else `medium`.

In [ ]:
from src.stores import learner
print("mastery now:", learner.all_mastery("notebook-demo"))
print("difficulty_for:", learner.difficulty_for("notebook-demo", "photosynthesis"))
for i, correct in enumerate([True, True, False, True]):
    p = learner.update("notebook-demo", "photosynthesis", correct)
    print(f"step {i+1} correct={correct} -> p={p:.3f}")

## 9. Eval — simulator (always runs) + RAGAS (needs keys/testset)
`simulate()`: 3 profiles (strong .85 / avg .6 / weak .35) x N sessions through G2+G3 → `gain` + `repetition_rate`.

In [ ]:
from src.eval.evaluate import simulate, ragas_eval
print(json.dumps(simulate(sessions=2), indent=2))
print(json.dumps(ragas_eval(), indent=2)[:500])

## 10. Serving — FastAPI + Streamlit are thin wrappers
- `POST /ingest` → `ingest_file()` · `POST /chat` → `ask()` · `POST /quiz` → quiz graph · `GET /mastery/{id}` → `all_mastery()`
- Streamlit tabs: Upload(G0) | Tutor(G1) | Quiz(G2) | Dashboard(G3)

In [ ]:
from src.api.main import app
print("routes:", [(r.path, sorted(r.methods)) for r in app.routes if hasattr(r, 'path')])
print("\nRun API: uvicorn src.api.main:app --reload")
print("Run UI : streamlit run src/ui/app.py")

## Summary + next steps
You just ran **smoke.py's full workflow live**: ingest → ask → refuse-test → quiz → feedback → BKT → simulate.

**Try next:**
1. `ingest_file("data/uploads/<your.pdf>")` with your own material
2. `ask("...", topic="photosynthesis")` with a topic filter
3. Quiz with `difficulty_target="auto"` — difficulty adapts after graded answers
4. Inspect `data/checkpoints/bkt.json` (mastery) and `data/chroma_db/` (vectors)
5. Add `GOOGLE_API_KEY` + `OPENAI_API_KEY` to `.env` to enable LLM tagging/answering/verification